# Thêm các thư viện

In [12]:
import os
import sys
import torch
import yaml
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

if os.path.basename(os.getcwd()) != 'audio_event_detection':
    os.chdir('..')
print("Thư mục hiện tại:", os.getcwd())

sys.path.insert(0, os.getcwd())

from models.ast_model import AudioSpectrogramTransformer
from scripts.train import Trainer
from utils.dataset import AudioEventDataset

# Khóa Random State
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)


Thư mục hiện tại: h:\audio_event_detection


In [13]:
config_path = 'configs/config.yaml'
csv_path = 'data/processed/spectrograms/processed_metadata.csv'

with open(config_path, 'r') as f:
    config_dict = yaml.safe_load(f)

batch_size = config_dict['training']['batch_size']
num_workers = config_dict.get('hardware', {}).get('num_workers', 4)
pin_memory = config_dict.get('hardware', {}).get('pin_memory', True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sử dụng thiết bị phần cứng: {device}")


Sử dụng thiết bị phần cứng: cpu


# Chia train/val/test

In [14]:
metadata_df = pd.read_csv(csv_path)

# Xóa các samples bị lỗi nhãn
metadata_df = metadata_df.dropna(subset=['label'])
metadata_df['label'] = metadata_df['label'].astype(int)

print(f"\nPhân phối Label trong bộ Dataset:")
print(metadata_df['target_class'].value_counts())

# Tách dữ liệu: 80% Train, 10% Val, 10% Test
train_df, temp_df = train_test_split(metadata_df, test_size=0.2, random_state=42, stratify=metadata_df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"\nChia dữ liệu thành công!")
print(f"Số lượng Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")



Phân phối Label trong bộ Dataset:
target_class
normal            8309
dog_bark          1000
siren              929
gunshot            374
explosion           40
fire_crackling      40
scream              40
Name: count, dtype: int64

Chia dữ liệu thành công!
Số lượng Train=8585 | Val=1073 | Test=1074


# Tạo DataLoader

In [15]:
train_dataset = AudioEventDataset(train_df, config_path, mode='train')
val_dataset = AudioEventDataset(val_df, config_path, mode='val')

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                          num_workers=num_workers, pin_memory=pin_memory, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                        num_workers=num_workers, pin_memory=pin_memory)
                         
print("Tạo DataLoaders Hoàn tất!")


Tạo DataLoaders Hoàn tất!


# Tạo Trainer

In [16]:
model = AudioSpectrogramTransformer(config_path)

# Nếu bạn muốn kiểm tra số lượng Parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Số lượng tham số mô hình: {total_params:,}")

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config_path=config_path,
    device=str(device)
)

Số lượng tham số mô hình: 85,414,664
Trainer initialized
Model parameters: 85,414,664
Training samples: 8585
Validation samples: 1073


# Bắt đầu huấn luyện

In [ ]:
print("\n🚀 Bắt đầu quá trình Huấn Luyện...")
trainer.train()


Trainer initialized
Model parameters: 85,414,664
Training samples: 8585
Validation samples: 1073
